# Resource Estimation for Simulating a 2D Ising Model Hamiltonian

In this notebook we demonstrate how to estimate the resources for quantum
dynamics, specifically the simulation of an Ising model Hamiltonian on an
$N \times N$ 2D lattice using a *fourth-order Trotter–Suzuki product formula*
assuming a 2D qubit architecture with nearest-neighbor connectivity.

We use **qdk-chemistry** for domain-specific functionality (model
Hamiltonians, Trotter decomposition, circuit construction) and
**qsharp.qre** exclusively for quantum resource estimation.

## Background: 2D Ising Model

The Ising model is a mathematical model of ferromagnetism in a lattice
(in our case a 2D square lattice) with two kinds of terms in the
Hamiltonian: (i) an interaction term between adjacent sites and (ii) an
external magnetic field acting at each site. We consider a simplified
version where the interaction terms have the same strength and the
external field strength is the same at each site.

Formally, the Ising model Hamiltonian on an $N \times N$ lattice is:

$$
H = \underbrace{-J \sum_{\langle i, j \rangle} Z_i Z_j}_{B} - \underbrace{h \sum_j X_j}_{A}
$$

where $J$ is the interaction strength and $h$ is the external field strength.

## Creating the Ising Model

We use `qdk_chemistry` to define the lattice geometry and build the
Hamiltonian.  `LatticeGraph.square` creates a 2D square lattice and
`create_ising_hamiltonian` produces the corresponding `QubitHamiltonian`
with ZZ interaction and transverse X field terms.

In [ ]:
from qdk_chemistry.data import LatticeGraph
from qdk_chemistry.utils.model_hamiltonians import create_ising_hamiltonian

lattice = LatticeGraph.square(10, 10, t=1.0)
hamiltonian = create_ising_hamiltonian(lattice, j=1.0, h=0.5)

print(f"Ising model on {lattice.num_sites}-site lattice "
      f"with {lattice.num_edges} edges")
print(f"QubitHamiltonian: {len(hamiltonian.pauli_strings)} Pauli terms, "
      f"{hamiltonian.num_qubits} qubits")

## Trotter Expansion

The time evolution $e^{-iHt}$ for the Hamiltonian is simulated with the
fourth-order product formula so that any errors in simulation are
sufficiently small. Essentially, this is done by simulating the evolution
for small slices of time $\Delta$ and repeating this for `nSteps`
$= t/\Delta$ to obtain the full time evolution. The Trotter–Suzuki
formula for higher orders can be recursively defined using a *fractal
decomposition*. In particular, the fourth-order formula $U_4(\Delta)$
can be constructed using the second-order one $U_2(\Delta)$ as follows.

$$
\begin{aligned}
U_2(\Delta) & = e^{-iA\Delta/2} e^{-iB\Delta} e^{-iA\Delta/2}; \\
U_4(\Delta) & = U_2(p\Delta)U_2(p\Delta)U_2((1 - 4p)\Delta)U_2(p\Delta)U_2(p\Delta); \\
p & = (4 - 4^{1/3})^{-1}.
\end{aligned}
$$

The `Trotter` builder in qdk-chemistry implements arbitrary-order
Suzuki decompositions and automatically determines the number of
Trotter steps for a given target accuracy (or accepts an explicit
step count).

In [ ]:
import math
from qdk_chemistry.algorithms.time_evolution.builder.trotter import Trotter

t = 1.0   # total simulation time
dt = 0.5  # time per Trotter step

trotter = Trotter(order=4, num_divisions=math.ceil(t / dt), optimize_term_ordering=True)
evolution = trotter.run(hamiltonian, time=t)

container = evolution.get_container()
print(f"4th-order Trotter: {len(container.step_terms)} Pauli "
      f"exponentials per step, {container.step_reps} repetitions")
print(f"Qubits: {evolution.get_num_qubits()}")

## Building the Circuit

The `TimeEvolutionUnitary` is converted to a `Circuit` using the
`PauliSequenceMapper`, which compiles the Pauli product formula into
a Q# circuit with QIR. This circuit can be used directly for resource
estimation via `circuit.estimate()`.

In [ ]:
from qdk_chemistry.algorithms.time_evolution.circuit_mapper import PauliSequenceMapper

mapper = PauliSequenceMapper()
circuit = mapper.run(evolution)

print(f"Circuit has QIR: {circuit.qir is not None}")
print(f"Circuit has Q#:  {circuit.qsharp is not None}")

## Resource Estimation

The `Circuit` can estimate resources directly via `circuit.estimate()`,
which uses the Q# resource estimator under the hood.

In [ ]:
result = circuit.estimate()
print(f"Physical qubits: {result['physicalCounts']['physicalQubits']}")
print(f"Runtime: {result['physicalCounts']['runtime']}")

## Visualizing and Understanding the Results

### Result Summary Table

Canonically the number of physical qubits, runtime for a shot, and the
shot error rate are presented. QRE reports only on configurations of the
parameters that are Pareto-optimal in time and space (and satisfy the
specified error rate in the `estimate` command). We add additional columns
for information about these configurations.

Notice the tradeoff between space (number of qubits) and time (runtime) is
governed entirely by the selection of code distance. The higher code
distance means the algorithm runs more slowly and the compute register is
larger. But as a side effect we need far fewer magic state factories, and
hence the total qubit count goes down!

## Tradeoff Curves

We can use matplotlib to create nice plots of our tradeoff curve.